<h1 style="text-align: center;">Conclusion & Recommendation</h1>
<h3 style="text-align: center;">Hotel Booking Cancellation Prediction</h3>

---

<h5 style="text-align: right;">By Beta Group</h5>

# **Section 1: Conclusion**

## **1.1 Model**

* **Pemilihan Arsitektur Model Terbaik**:
    * **Model Terpilih**: `XGBoost` dipilih sebagai model final setelah melalui tahap benchmarking dan hyperparameter tuning karena menghasilkan performa validasi terbaik dibandingkan `Logistic Regression`.
    * **Hasil Pencarian Model**: Setelah `RandomizedSearchCV` menguji 2.000 kombinasi hyperparameter, XGBoost menghasilkan ROC-AUC training sebesar **0,9670** dan ROC-AUC validation sebesar **0,9351**. Logistic Regression menghasilkan ROC-AUC validation yang lebih rendah, yaitu **0,9108**.
    * **Stabilitas Validasi**: Selisih ROC-AUC training dan validation sebesar **3,19** percentage points, jauh di bawah batas gap yang ditetapkan pada tahap tuning (≤ 25 percentage points), sehingga tidak ditemukan indikasi overfitting berat pada tahap validasi.
* **Ringkasan Performa Final pada Data Test (Threshold = 0,7426)**:
    * **Kemampuan Diskriminasi**: ROC-AUC = **0,8861**
    * **Metrik Klasifikasi**: Accuracy = **79,92%** | Precision = **80,97%** | Recall = **62,97%** | F1-score = **70,84%**
    * **Confusion Matrix (n = 40.619):**
        * True Positive (TP): 9.908
        * False Positive (FP): 2.329
        * False Negative (FN): 5.826
        * True Negative (TN): 22.556
    * **Tingkat Kesalahan**: False Positive Rate = **9,36%** | False Negative Rate = **37,03%**
    * **Interpretasi Utama**: Dari setiap 100 booking yang diprediksi cancel, sekitar **81 booking** benar-benar cancel. Namun, dari setiap 100 cancellation aktual, model baru berhasil menemukan sekitar **63 booking**, sedangkan sekitar **37 booking masih terlewat**.
* **Scorecard terhadap Success Criteria (Business Understanding, Section 6)**:

| Kriteria | Target Resmi | Hasil | Status |
| :--- | :---: | :---: | :--- |
| **ROC-AUC** | ≥ 0,80 dan lebih tinggi dari baseline | **0,8861** (baseline default XGBoost = 0,7248) | Tercapai: model memiliki kemampuan diskriminasi yang baik dan mengungguli baseline |
| **Accuracy** | ≥ 75% | **79,92%** | Tercapai: hampir 80% seluruh booking diklasifikasikan dengan benar |
| **Generalization Check (No Significant Overfitting)** | Tidak ada gap signifikan train-validation (ambang batas yang dipakai pada model selection: gap ≤ 25 poin, ROC-AUC validasi ≥ 0,70) | Gap train-validation **3,19 poin** | Tercapai dengan margin besar |
| **Bebas Data Leakage** | Fitur hanya berasal dari informasi yang tersedia saat reservasi dibuat | `reservation_status`/`reservation_status_date` sudah dibuang, **namun `deposit_type` dan `room_type_changed` masih dipakai model dan sudah ditandai sebagai potensi leakage** | **Belum sepenuhnya tercapai** |

> **Catatan:** Precision (80,97%), Recall (62,97%), F1-score (70,84%), serta gap validation–test (4,90 poin) adalah metrik pendukung yang informatif secara bisnis, tetapi bukan merupakan Success Criteria resmi yang ditetapkan pada tahap Business Understanding sehingga tidak diberi label 'Tercapai/Belum Tercapai' terhadap target yang tidak pernah ditetapkan. Recall 62,97% tetap perlu dicatat sebagai keterbatasan praktis karena sekitar 37% cancellation aktual belum terdeteksi model.

* **Reliabilitas dan Kesiapan Implementasi**:

    * **Kemampuan Generalisasi & Indikasi Concept Drift**: ROC-AUC menurun secara monoton dari **0,9670 pada 2015**, **0,9352 pada 2016**, menjadi **0,8861 pada 2017 (test)**. Karena data 2015 dan 2016 sama-sama merupakan data *in-sample* yang dipakai untuk fitting model final, penurunan performa yang sudah terjadi antara kedua periode tersebut mengindikasikan bahwa gap performa **bukan semata-mata overfitting, melainkan mengandung komponen *concept drift*** yaitu perubahan pola hubungan antara karakteristik reservasi dan cancellation dari waktu ke waktu (Notebook 6 dan 7). Implikasinya, model bukan solusi *deploy-and-forget* dan memerlukan monitoring performa serta retraining berkala.
    * **Risiko Leakage yang Belum Terselesaikan**: Pengecekan feature alignment pada tahap implementasi (Notebook 7) mengonfirmasi bahwa `deposit_type` dan `room_type_changed`, dua fitur yang sejak tahap EDA dan interpretasi model sudah ditandai berpotensi leakage, **masih digunakan oleh model final dan masih tersedia di data test**. Akibatnya, ROC-AUC test sebesar 0,8861 berpotensi **overestimate** kemampuan model sebenarnya pada kondisi produksi, sampai kedua fitur ini diaudit dan dipastikan benar-benar tersedia pada saat prediksi harus dibuat (sebelum kedatangan tamu).
    * **Explainability**: Analisis feature importance (gain-based), SHAP, dan permutation importance secara konsisten mengidentifikasi kandidat *business driver* yang relatif aman dari leakage, yaitu `agent`, `country`, `lead_time`, `required_car_parking_spaces`, `total_of_special_requests`, dan `booking_changes`. Sebaliknya, `deposit_type` dan `room_type_changed` memang berkontribusi tinggi dan konsisten pada ketiga metode, tetapi kontribusinya **tidak dapat langsung dibaca sebagai business driver murni** karena keduanya membawa risiko leakage. `previous_cancellations` juga **tidak direkomendasikan** sebagai business insight karena rankingnya tinggi pada gain dan SHAP (data training) tetapi kontribusinya mendekati nol pada permutation importance di data test, pola yang konsisten dengan leakage pada fitur turunannya, `previous_cancellation_rate`.

## **1.2 Bisnis**

* **Jawaban Langsung atas Pertanyaan Bisnis Utama:**
    * **Profil Booking Prioritas**: Risiko cancellation tidak ditentukan oleh satu karakteristik saja, melainkan kombinasi riwayat booking, sumber reservasi, komitmen pelanggan, dan jarak waktu pemesanan. Booking di **City Hotel**, memiliki **lead time panjang**, berasal dari segmen **Groups**, dan menggunakan channel **TA/TO** menjadi kelompok yang perlu memperoleh perhatian lebih besar.
    * **Faktor Risiko Utama**: Analisis EDA dan model interpretation secara konsisten menunjukkan bahwa `agent`, `country`, `lead_time`, `required_car_parking_spaces`, `total_of_special_requests`, dan `booking_changes` memberikan kontribusi terhadap prediksi model **tanpa indikasi leakage**. `deposit_type` dan `room_type_changed` juga berkontribusi besar terhadap prediksi, tetapi keduanya berstatus potensi data leakage (lihat Section 1.1) sehingga tidak dapat langsung diperlakukan sebagai business driver yang berdiri sendiri sebelum diaudit lebih lanjut. Secara umum, kontribusi fitur menunjukkan pola yang digunakan model dan tidak membuktikan bahwa fitur tersebut secara langsung menyebabkan cancellation.
    * **Prioritas Intervensi**: Model memberikan alarm kepada **12.237 dari 40.619 booking** pada data test, terbagi menjadi 14.544 booking Low Risk, 13.838 booking Medium Risk, dan 12.237 booking High Risk. Dari 12.237 booking yang dialarm, **9.908 booking benar-benar cancel**, sehingga precision mencapai **80,97%**. Hal ini menunjukkan bahwa model dapat membantu hotel mempersempit sasaran intervensi kepada kelompok yang lebih relevan.
    * **Dampak Utama**: Penggunaan model berpotensi meningkatkan ketepatan occupancy forecasting, membantu pengelolaan inventory kamar, dan memberikan waktu lebih awal untuk melakukan tindakan preventif. Simulasi cost-benefit awal menunjukkan indikasi kelayakan ekonomi yang kuat, tetapi keuntungan finansial aktual tetap belum dapat dinyatakan final sebelum efektivitas intervensi dan biaya program diuji menggunakan data operasional hotel.
* **Penyelesaian Tantangan Operasional**:
    * **Targeted Intervention**: Model membantu menjawab pertanyaan **booking mana yang perlu diprioritaskan** dengan mengurutkan reservasi berdasarkan probabilitas cancellation. Dengan pendekatan ini, tim hotel tidak perlu menghubungi seluruh tamu dan dapat memusatkan sumber daya pada booking dengan risiko serta nilai reservasi yang lebih tinggi.
* **Lead Time sebagai Peluang Intervensi Utama**:
    * **Temuan:** Cancellation rate meningkat secara konsisten seiring bertambahnya lead time, mulai dari **9,6% pada booking 0–7 hari hingga 67,7% pada booking di atas 365 hari**. Booking dengan lead time lebih dari 180 hari juga memiliki tingkat cancellation yang jauh lebih tinggi daripada booking jangka pendek.
    * **Batas Interpretasi**: Lead time tidak boleh digunakan sebagai satu-satunya dasar intervensi. Keputusan tetap harus mempertimbangkan probabilitas model, nilai booking, deposit type, market segment, serta biaya tindakan.
* **Prioritas Berdasarkan Hotel, Segmen, dan Channel**:
    * **Temuan Hotel**: City Hotel memiliki cancellation rate **41,8%**, lebih tinggi daripada Resort Hotel sebesar **27,8%.** City Hotel juga memiliki volume booking lebih besar sehingga menjadi area prioritas untuk pilot implementasi.
    * **Temuan Segmen:** Market segment **Groups** memiliki cancellation rate **61,1%** dan menjadi segmen berisiko tinggi yang memiliki volume material. Sementara itu, **Online TA** memiliki cancellation rate **36,8%**, tetapi volumenya yang besar membuatnya tetap menjadi salah satu kontributor utama jumlah cancellation.
    * **Temuan Channel**: Channel **TA/TO** mencakup sekitar **82% dari seluruh booking** dan memiliki cancellation rate **41,1%,** lebih tinggi daripada overall cancellation rate sebesar **37,08%.**
* **Anomali Kebijakan Deposit sebagai Area Validasi:**
    * **Temuan:** Kategori `Non Refund` memiliki cancellation rate ekstrem sebesar **99,4%**, jauh lebih tinggi dibandingkan `No Deposit` sebesar **28,4%** dan `Refundable` sebesar **22,2%**. Pola ini berlawanan dengan dugaan bahwa kebijakan `non-refundable` seharusnya menurunkan cancellation, dan `deposit_type` juga sudah ditandai sebagai fitur dengan potensi leakage pada tahap model interpretation.
    * **Risiko Interpretasi**: Pola tersebut mungkin berkaitan dengan agent tertentu, market segment, country, proses pencatatan, atau waktu penetapan status deposit. Oleh karena itu, hasil ini belum membuktikan bahwa kebijakan Non Refund menyebabkan cancellation, dan kontribusinya yang tinggi terhadap model perlu dibaca bersama catatan leakage di atas.
* **Kesenjangan Kualitas dan Tata Kelola Data**:
    * **Temuan**: Dataset tidak memiliki ID reservasi unik. Pada data mentah ditemukan **31.994 baris identik**, dan setelah tahap cleaning jumlah baris identik pada dataset final (119.205 baris) justru sedikit meningkat menjadi **32.238 baris**. Karena dua booking yang berbeda masih mungkin memiliki karakteristik sama, baris tersebut tidak dapat langsung dinyatakan sebagai duplikat yang harus dihapus.
    * **Temuan Missing Value**: `company` dan `agent` memiliki banyak nilai kosong yang kemungkinan berarti booking tidak menggunakan perusahaan atau agen. Sementara itu, missing value pada `country` tidak tersebar secara acak dan lebih banyak ditemukan pada channel Corporate dan Direct.
* **Validasi Dampak Finansial**:
    * **Temuan Revenue Exposure**: Total sebesar **€7.223.580** merupakan estimasi nilai booking cancellation pada data test berdasarkan ADR dan lama menginap (`adr` × `total_stay_nights`), bukan kerugian bersih.
    * **Simulasi Cost-Benefit**: Dengan asumsi biaya intervensi sebesar **€1,11 per booking yang dihubungi** (estimasi dari biaya waktu staf, biaya telepon, dan durasi kontak 3 menit), total biaya untuk menghubungi seluruh **12.237 booking** yang dialarm model adalah **€13.583**. Pada skenario success rate intervensi 30%, 50%, dan 70%, estimasi revenue yang dapat dipertahankan (net setelah biaya intervensi) masing-masing sekitar **€1.268.543**, **€2.123.293**, dan **€2.978.043**. **Break-even success rate hanya sebesar 0,32%**, artinya secara matematis intervensi hampir pasti menguntungkan bahkan dengan asumsi keberhasilan yang sangat rendah.
    * **Batas Interpretasi**: Meskipun break-even rate terlihat sangat rendah, hasil simulasi ini **sensitif terhadap dua asumsi yang belum divalidasi**: 
        1. Success rate intervensi (30/50/70%) masih berupa skenario, belum didukung pilot test
        2. Potential loss dari `adr × total_stay_nights` belum memperhitungkan kemungkinan kamar dijual kembali, variasi harga musiman, atau tamu yang reschedule alih-alih benar-benar cancel. Angka ini lebih tepat dibaca sebagai **indikasi awal kelayakan ekonomi**, bukan estimasi final penghematan, dan belum memperhitungkan cancellation fee maupun biaya implementasi model.

# **Section 2. Recommendation**

## **2.1 Model**
Untuk mengimplementasikan model secara aman dan menghasilkan dampak bisnis yang terukur, beberapa langkah berikut direkomendasikan:

1. **Gunakan sebagai Decision-Support System**:

    * Model belum disarankan untuk mengambil keputusan secara otomatis, tetapi sudah layak digunakan dalam **pilot terbatas sebagai decision-support system** untuk membantu Revenue Manager dan Reservation Team memprioritaskan booking berisiko.
    * Prediksi cancel hanya merupakan peringatan risiko. Hotel tidak boleh langsung membatalkan reservasi, memberikan refund, menjual ulang kamar, atau melakukan overbooking hanya berdasarkan output model.
    * Keputusan pembatalan, refund, resale, dan overbooking tetap berada pada Revenue Manager.

2. **Sesuaikan Threshold dengan Tujuan Bisnis**:

    * Threshold 0,7426 dipilih dengan memaksimalkan **Accuracy** pada data validation, bukan hasil optimisasi biaya-manfaat bisnis, sehingga angka ini tidak perlu dijadikan permanen.
    * Naikkan threshold untuk mengurangi intervensi salah sasaran atau turunkan untuk menangkap lebih banyak cancellation (saat ini 37% cancellation aktual masih lolos sebagai False Negative).
    * Pertimbangkan biaya False Positive, False Negative, biaya intervensi (€1,11/booking), dan nilai booking, mengingat model saat ini memakai satu threshold yang sama untuk dua kebutuhan berbeda: soft preventive action dan dukungan keputusan overbooking.

3. **Audit Fitur dan Selesaikan Leakage Sebelum Deployment Penuh**:

    * **Prioritas utama**: `deposit_type` dan `room_type_changed` masih digunakan oleh model final dan masih tersedia di data test meskipun sudah ditandai berpotensi leakage sejak tahap EDA dan interpretasi model. Pastikan status ketersediaan kedua fitur ini pada saat prediksi benar-benar dibutuhkan (sebelum kedatangan tamu) sebelum mengandalkannya lebih lanjut; jika tidak tersedia pada waktu tersebut, model perlu dilatih ulang tanpa fitur ini dan performa dievaluasi ulang.
    * Investigasi juga divergensi `previous_cancellations`/`previous_cancellation_rate` (tinggi pada training, mendekati nol pada permutation importance di test) sebelum kedua fitur ini digunakan sebagai dasar business insight.
    * Tambahkan calibration curve dan Brier score agar risk score dapat diinterpretasikan dengan tepat.

4. **Perbaiki Tata Kelola Data**:

    * Sediakan booking ID yang konsisten dan definisi data yang terdokumentasi, termasuk waktu pencatatan setiap fitur.
    * Gunakan kategori eksplisit seperti `Unknown`, `No Agent`, dan `No Company`, alih-alih membiarkan nilai kosong tanpa keterangan.

5. **Evaluasi dan Monitor Secara Berkala**

    * Periksa performa berdasarkan hotel, segmen, channel, musim, dan lead time.
    * Lakukan monitoring dan retraining berkala karena ROC-AUC menurun secara monoton dari **0,9670 (2015) => 0,9352 (2016) => 0,8861 (2017/test)**, pola yang mengindikasikan *concept drift* (perubahan pola cancellation dari waktu ke waktu), bukan sekadar overfitting klasik. Model dilatih pada data 2015–2017 dari dua hotel di Portugal sehingga performanya pada kondisi saat ini (termasuk perubahan pola pasca-pandemi) tidak dapat diasumsikan setara dan wajib divalidasi ulang dengan data terbaru sebelum deployment produksi.


## **2.2 Bisnis**
1. **Fokuskan intervensi awal** pada City Hotel, lead time panjang, segmen Groups, dan channel TA/TO yang menunjukkan risiko cancellation material. Untuk booking dengan lead time panjang, berikan reminder atau konfirmasi tambahan secara bertahap, terutama mendekati batas free cancellation dan tanggal kedatangan. Intervensi tetap diberikan pada tingkat reservasi berdasarkan risk score, bukan hanya berdasarkan keanggotaan segmen.

2. **Terapkan tindakan sesuai tingkat risiko:** booking Low Risk (14.544 booking pada data test) cukup dipantau secara rutin, Medium Risk (13.838 booking) menerima reminder otomatis / konfirmasi ringan, sedangkan booking High Risk (12.237 booking) langsung menerima rekomendasi aksi administratif (reconfirmation, status refund, dan resale kamar) berdasarkan `deposit_type` booking tersebut. Prediksi tidak digunakan untuk membatalkan reservasi secara otomatis.

3. **Validasi anomali Non Refund** sebelum mengubah kebijakan deposit karena cancellation rate 99,4% dapat dipengaruhi agent, segmen, atau proses pencatatan, dan `deposit_type` sendiri masih berstatus potensi data leakage pada model. Lakukan pemeriksaan silang terhadap agent, country, market segment, distribution channel, dan waktu pencatatan deposit sebelum mengubah kebijakan.

4. **Validasi asumsi cost-benefit sebelum scale-up**: simulasi awal menunjukkan break-even success rate intervensi hanya 0,32% dari total biaya sekitar €13.583 untuk 12.237 booking berisiko, indikasi kelayakan ekonomi yang kuat. Namun, asumsi success rate intervensi (30/50/70%) dan estimasi potential loss (`adr` × lama menginap) belum divalidasi dengan data operasional aktual. Jalankan pilot terbatas (misalnya di City Hotel) untuk mengukur success rate intervensi yang sesungguhnya sebelum mengklaim penghematan finansial secara pasti.